# Step 3 of 3 — Exploratory Data Analysis
**Pipeline:** `01_data_cleaning` → `02_sql_analysis` → `03_eda`

**Reads:**
- `output/customer_behavior_clean.csv` (Step 1)
- `output/customer_behavior_treated.csv` (Step 1 — outliers capped)
- `output/sql_results/*.csv` (Step 2 — SQL query results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import config

sns.set_theme(style="whitegrid")

config.require(config.CLEAN_CSV, "01_data_cleaning.ipynb")
config.require(config.TREATED_CSV, "01_data_cleaning.ipynb")
config.require(config.SQL_OUT_DIR / "kpis.csv", "02_sql_analysis.ipynb")

df = pd.read_csv(config.CLEAN_CSV, parse_dates=["visit_date"])           # real values
df_clean = pd.read_csv(config.TREATED_CSV, parse_dates=["visit_date"])   # outliers capped

sql = {p.stem: pd.read_csv(p) for p in config.SQL_OUT_DIR.glob("*.csv")}
print("clean:", df.shape, "| treated:", df_clean.shape, "| SQL results:", list(sql))

## 1. Dashboard from SQL results (Step 2)

In [ ]:
print("DASHBOARD KPIs")
display(sql["kpis"].T.rename(columns={0: "value"}).round(2))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.lineplot(data=sql["monthly_revenue"], x="month", y="revenue", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Revenue by Month"); axes[0, 0].tick_params(axis="x", rotation=45)

sns.barplot(data=sql["session_outcome"], x="session_outcome", y="total_sessions", ax=axes[0, 1])
axes[0, 1].set_title("Session Outcome"); axes[0, 1].tick_params(axis="x", rotation=25)

sns.barplot(data=sql["category_revenue"], x="revenue", y="product_category", ax=axes[0, 2])
axes[0, 2].set_title("Revenue by Category")

sns.barplot(data=sql["state_revenue"], x="revenue", y="state", ax=axes[1, 0])
axes[1, 0].set_title("Revenue by State")

sns.barplot(data=sql["payment_analysis"], x="payment_method", y="total_purchases", ax=axes[1, 1])
axes[1, 1].set_title("Purchases by Payment Method"); axes[1, 1].tick_params(axis="x", rotation=25)

sns.barplot(data=sql["device_sessions"], x="device_type", y="total_sessions", ax=axes[1, 2])
axes[1, 2].set_title("Sessions by Device")

plt.suptitle("Indian E-commerce Customer Behavior Dashboard", fontsize=16)
plt.tight_layout()
plt.show()

## 2. Univariate analysis — numerical columns (outliers, skewness, distribution)

In [ ]:
num_plan = {  # column: (title, discrete histogram?)
    "customer_age":     ("Customer Age", False),
    "unit_price_inr":   ("Unit Price", False),
    "quantity":         ("Quantity", True),
    "discount_percent": ("Discount Percentage", True),
    "revenue_inr":      ("Revenue", False),
}

summary = []
for col, (title, discrete) in num_plan.items():
    data = df[col].dropna()
    q1, q3 = data.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((data < lower) | (data > upper)).sum()
    skew = data.skew()
    shape = ("positively (right) skewed" if skew > 0.5
             else "negatively (left) skewed" if skew < -0.5 else "approximately symmetric")
    summary.append({"column": col, "outliers": n_out, "skewness": round(skew, 2), "shape": shape})

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    if discrete:
        sns.histplot(data=data, discrete=True, ax=axes[0])
    else:
        sns.histplot(data=data, kde=True, stat="density", ax=axes[0])
    axes[0].set_title(f"{title}: Distribution")
    sns.boxplot(x=data, ax=axes[1]);            axes[1].set_title(f"{title}: Outliers")
    stats.probplot(data, dist="norm", plot=axes[2]); axes[2].set_title(f"{title}: Normality Q-Q Plot")
    plt.tight_layout()
    plt.show()

display(pd.DataFrame(summary))

## 3. Univariate analysis — categorical columns

In [ ]:
cat_cols = ["city", "state", "customer_type", "device_type",
            "marketing_channel", "product_category", "payment_method"]

for col in cat_cols:
    data = df[col].fillna("Missing").astype(str)
    freq = data.value_counts()
    table = pd.DataFrame({"count": freq, "percent": (freq / len(data) * 100).round(2)})
    print(f"\n=== {col} === (missing: {df[col].isna().sum()}, mode: {data.mode().tolist()})")
    display(table.head(15))

    plt.figure(figsize=(11, 4.5))
    sns.countplot(data=df, x=col, order=freq.head(15).index)
    plt.title(f"{col}: Category Distribution (Top 15)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 4. Bivariate analysis

In [ ]:
def purchase_split(col, rotate=0, target="purchased"):
    sns.countplot(data=df, x=col, hue=target)
    plt.title(f"{col} vs {target}")
    plt.xticks(rotation=rotate)
    plt.show()
    rate = df.groupby(col)[target].mean().mul(100).round(2).sort_values(ascending=False)
    print(f"{target} rate (%) by {col}:")
    print(rate.to_string())
    print(f"→ Highest: {rate.idxmax()} ({rate.max()}%) | Lowest: {rate.idxmin()} ({rate.min()}%)\n")

purchase_split("customer_type", 30)
purchase_split("device_type")
purchase_split("marketing_channel", 45)
purchase_split("device_type", target="cart_abandoned")

In [ ]:
revenue = df.groupby("product_category")["revenue_inr"].sum().sort_values(ascending=False)
revenue.plot(kind="bar")
plt.title("Product Category vs Revenue"); plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
print(revenue.round(2).to_string())
print(f"→ Top revenue category: {revenue.idxmax()}")

In [ ]:
sns.boxplot(data=df, x="purchased", y="discount_percent")
plt.title("Discount vs Purchase"); plt.show()
print(df.groupby("purchased")["discount_percent"].mean().round(2))

sns.scatterplot(data=df, x="quantity", y="revenue_inr")
plt.title("Quantity vs Revenue"); plt.show()
print("Correlation (quantity, revenue):", round(df["quantity"].corr(df["revenue_inr"]), 3))

## 5. Correlation heatmap

In [ ]:
corr_cols = [c for c in ["customer_age", "unit_price_inr", "quantity", "discount_percent",
                         "revenue_inr", "purchased", "cart_abandoned"] if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("E-commerce Numerical Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Validation of the treated (outlier-capped) data from Step 1

In [ ]:
print("Treated data shape:", df_clean.shape)
print("Missing values left:", df_clean.isnull().sum().sum())
print("Duplicate rows     :", df_clean.duplicated().sum())

rows = []
for col in num_plan:
    s = df_clean[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    rows.append({
        "column": col,
        "outliers before": int(((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()),
        "outliers after":  int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()),
        "skew before": round(df[col].skew(), 2),
        "skew after":  round(s.skew(), 2),
    })
display(pd.DataFrame(rows))

## 7. Final business insights

In [ ]:
# Insights use the clean (uncapped) data so revenue totals stay real
print("=" * 50)
print("E-COMMERCE EDA: BUSINESS INSIGHTS")
print("=" * 50)

print(f"\n1. Purchase Rate: {df['purchased'].mean() * 100:.2f}%")

cat_rev = df.groupby("product_category")["revenue_inr"].sum().sort_values(ascending=False)
print(f"\n2. Top Revenue Category: {cat_rev.idxmax()} (INR {cat_rev.max():,.0f})")

ch = df.groupby("marketing_channel")["purchased"].mean().mul(100).sort_values(ascending=False)
print("\n3. Purchase Rate by Marketing Channel (%):"); print(ch.round(2).to_string())
print("   Best converting channel:", ch.idxmax())

dv = df.groupby("device_type")["purchased"].mean().mul(100).sort_values(ascending=False)
print("\n4. Purchase Rate by Device (%):"); print(dv.round(2).to_string())
print("   Best converting device:", dv.idxmax())

st = df.groupby("state")["revenue_inr"].sum().sort_values(ascending=False)
print("\n5. Top 5 States by Revenue:"); print(st.head(5).round(2).to_string())

print(f"\n6. Cart Abandonment Rate: {df['cart_abandoned'].mean() * 100:.2f}%")
print(f"\n7. Average Revenue per Session: INR {df['revenue_inr'].mean():,.2f}")

# Cross-check: the Python numbers must match the SQL KPIs from Step 2
k = sql["kpis"].iloc[0]
print("\n--- Cross-check with SQL (Step 2) ---")
print("Total revenue  | SQL:", round(k["total_revenue"], 2), "| pandas:", round(df["revenue_inr"].sum(), 2))
print("Conversion %   | SQL:", round(k["conversion_rate"], 2), "| pandas:", round(df["purchased"].mean() * 100, 2))
print("\n✔ Pipeline complete.")